In [ ]:
!pip install transformers
!pip install "datasets<2.19"
!pip install librosa

datasets

https://huggingface.co/datasets/google/fleurs

models

https://huggingface.co/facebook/wav2vec2-base

https://huggingface.co/facebook/wav2vec2-large-xlsr-53

https://huggingface.co/facebook/wav2vec2-xls-r-300

In [ ]:
"""
Phase 0: Wav2vec2 Phonological Feature Probing - Simple Version
"""

import torch
import numpy as np
from datasets import load_dataset
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
from tqdm import tqdm
import pickle
import os

# ============================================================================
# SETTINGS
# ============================================================================

MODEL_NAME = "facebook/wav2vec2-base"
LANGUAGES = ["en_us", "de_de", "es_419"]  # Start with 3 languages
MAX_SAMPLES = 100  # Samples per language for MVP
OUTPUT_DIR = "./extracted_features"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================================
# LOAD MODEL
# ============================================================================

print(f"Loading model: {MODEL_NAME}")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
model = Wav2Vec2Model.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()

print(f"Model loaded on {DEVICE}")
print(f"Number of layers: {model.config.num_hidden_layers}")
print(f"Hidden size: {model.config.hidden_size}")

# ============================================================================
# EXTRACT REPRESENTATIONS
# ============================================================================

def extract_representations(audio_array, sampling_rate):
    """Extract hidden states from all layers"""

    # Preprocess audio
    inputs = feature_extractor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt",
        padding=True
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    # Get representations from all layers
    with torch.no_grad():
        outputs = model(
            inputs["input_values"],
            output_hidden_states=True
        )

    # Convert to numpy (13 layers: CNN + 12 transformer)
    hidden_states = [h.cpu().numpy() for h in outputs.hidden_states]

    return hidden_states

# ============================================================================
# PROCESS DATA
# ============================================================================

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Process each language
for lang in LANGUAGES:
    print(f"\n{'='*60}")
    print(f"Processing language: {lang}")
    print(f"{'='*60}")

    # Load dataset
    print(f"Loading FLEURS dataset...")
    dataset = load_dataset(
        "google/fleurs",
        lang,
        split="train",
        trust_remote_code=True
    )

    # Limit samples
    if len(dataset) > MAX_SAMPLES:
        dataset = dataset.select(range(MAX_SAMPLES))

    print(f"Processing {len(dataset)} samples...")

    # Extract features for all samples
    all_features = []

    for idx, sample in enumerate(tqdm(dataset)):
        try:
            # Get audio
            audio = sample['audio']['array']
            sampling_rate = sample['audio']['sampling_rate']

            # Extract representations
            hidden_states = extract_representations(audio, sampling_rate)

            # Store everything
            feature_dict = {
                'id': sample['id'],
                'language': lang,
                'transcription': sample['transcription'],
                'raw_transcription': sample.get('raw_transcription', ''),
                'hidden_states': hidden_states,  # List of 13 arrays
                'audio_length': len(audio) / sampling_rate
            }

            all_features.append(feature_dict)

        except Exception as e:
            print(f"\nError on sample {idx}: {e}")
            continue

    # Save features
    output_file = f"{OUTPUT_DIR}/{lang}_features.pkl"
    with open(output_file, 'wb') as f:
        pickle.dump(all_features, f)

    print(f"\nSaved {len(all_features)} samples to {output_file}")

    # Print sample info
    if len(all_features) > 0:
        sample = all_features[0]
        print(f"\nSample structure:")
        print(f"  Transcription: {sample['transcription'][:50]}...")
        print(f"  Audio length: {sample['audio_length']:.2f}s")
        print(f"  Layer shapes:")
        for i, h in enumerate(sample['hidden_states']):
            print(f"    Layer {i}: {h.shape}")

print(f"\n{'='*60}")
print("Done! Features saved to:", OUTPUT_DIR)
print(f"{'='*60}")

Loading model: facebook/wav2vec2-base


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on cpu
Number of layers: 12
Hidden size: 768

Processing language: en_us
Loading FLEURS dataset...


Generating train split: 0 examples [00:00, ? examples/s]

...

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Processing 100 samples...


100%|██████████| 100/100 [05:22<00:00,  3.23s/it]



Saved 100 samples to ./extracted_features/es_419_features.pkl

Sample structure:
  Transcription: los murales o garabatos indeseados reciben el nomb...
  Audio length: 5.76s
  Layer shapes:
    Layer 0: (1, 287, 768)
    Layer 1: (1, 287, 768)
    Layer 2: (1, 287, 768)
    Layer 3: (1, 287, 768)
    Layer 4: (1, 287, 768)
    Layer 5: (1, 287, 768)
    Layer 6: (1, 287, 768)
    Layer 7: (1, 287, 768)
    Layer 8: (1, 287, 768)
    Layer 9: (1, 287, 768)
    Layer 10: (1, 287, 768)
    Layer 11: (1, 287, 768)
    Layer 12: (1, 287, 768)

Done! Features saved to: ./extracted_features
